# AI podcast generation: the full flow, runnable

This notebook walks one episode through the pipeline described in the PRD
(*AI Podcast Creation with Dynamic Content*): **8 stages, 1 fact-check gate, 1 rewrite loop**,
then the **dynamic refresh** that regenerates only what changed, and finally a **scorecard**
against the PRD's success metrics and requirements (FR1–FR13).

```
1 Ingest → 2 Index → 3 Retrieve and plan → 4 Script ⇄ 5 Fact-check → 6 Voice → 7 Assemble → 8 Publish
```

## What it does by default

It is set up for the real workflow: it **scrapes a web page** (the ClickHouse docs intro), has **Gemini**
write and fact-check the script, and **voices it with your cloned voice** (Chatterbox, using the recording
and consent record in `config/voices.yaml`). The *Your run* cell below holds those choices.

**Before you press Run All you need:**

| You need | Where |
|---|---|
| A Gemini API key | `secrets/llm_api_key.txt` (one line, git-ignored), or `GEMINI_API_KEY` set before the kernel starts. Create one at <https://aistudio.google.com/apikey> |
| The cloning kernel | `.venv-cloning`, made with `pip install -e ".[notebook,cloning]"` |
| Your voice recording and consent record | `assets/voices/` and `config/voices.yaml` (a clone without a consent record is refused) |
| A network | to fetch the page and to call Gemini |

The first cell checks all of this and stops with a plain message if something is missing, before any slow work.

**No key, no network, or no recording?** Set `PRESET = "samples"` in the *Your run* cell. That uses six bundled
demo texts, and `USE_LLM=false` / `TTS_BACKEND="say"` or `"placeholder"` swap in offline stand-ins. The flow is
the same; only the model and the voices are replaced.

| Piece | This notebook's default | Alternatives |
|---|---|---|
| Outline, script, fact-check judge | Gemini `gemini-3.8-flash` for every call, answering in JSON that matches a schema. If it is overloaded (503) the notebook waits and retries, for up to about two minutes per call | Claude (`LLM_PROVIDER=anthropic`); an offline extractive writer and word-overlap judge (`USE_LLM=false`) |
| Embeddings | Hashed bag-of-words vectors (offline) | `BAAI/bge-small-en-v1.5` (`USE_BGE = True`) |
| Voices | Chatterbox: Alex is your cloned voice, Sam is Chatterbox's default voice | macOS `say` system voices; placeholder tones |
| Sources | One web page, with each section its own source | The bundled sample texts; several URLs; live RSS (`LIVE_SOURCES = True`) |

**What is real here:** the control flow, the queue routing and the helper functions are imported from
`podcast_engine` itself (hashing, chunking, freshness scoring, page extraction, the fact-check routing rule, the
voice registry, the TTS cache key, the loudness check, the AI disclosure). **What is a prototype:** the stage
bodies, which in the repository are still unwritten stubs. Treat this notebook as the executable spec for them.

### Run it in VS Code or Jupyter

1. **Pick the kernel.** Click *Select Kernel* (top right) → *Python Environments* → **`.venv-cloning`**. That
   environment has the voice-cloning model; the plain `.venv` does not, and the notebook will say so in the
   first cell. (If `.venv-cloning` is not listed, reload the window, or see `docs/notebook.md`.)
2. **Choose what to run** in the *Your run* cell below: the page, the topic, how many segments, the voices, and
   whether the model is used (`USE_LLM`). **After editing it, use Run All** (or restart the kernel and run from the top).
   Re-running only some cells would keep the old settings, so the notebook stops with a message if you do.
3. **Run All.** Everything up to the voice step takes a minute or so (it includes the Gemini calls). The voice
   step takes about 10 minutes for this page and prints one line per sentence as it goes.
4. **Check that it is your voice.** The cell after the voice step plays your recording, then a line from each
   host, and scores how close each host is to your recording. The episode itself plays in section 8.
5. Files are in `notebooks/output/<run>/`: the audio, the show notes, the RSS feed and `run_manifest.json`.

The saved notebook keeps its outputs, including the audio, which makes the file large. Run `make
notebook-clean` before committing it. If you change this file outside the editor, revert the open tab
(*File → Revert File*) rather than saving it, or the editor's old copy will overwrite the change.

Every stage cell ends by showing its output, so you can inspect it.

## 0. Your run, setup and switches

**Your run** is the cell right below. The cell after it holds every other switch. Environment variables of the form
`PODCAST_NB_<NAME>` override both (the automated tests use them to run offline).

In [ ]:
import os

# ===== YOUR RUN: edit this cell ==================================================================
PRESET = "clickhouse"  # "clickhouse" = the web page below, voiced with config/voices.yaml
#                        "samples"    = the bundled demo texts, with whatever voices are available
PRESETS = {
    "clickhouse": {
        "SOURCE_URLS": "https://clickhouse.com/docs/get-started/about/intro",  # comma-separate for several
        "TOPIC": "What ClickHouse is and why it is fast for analytics",
        "TOPIC_KEYWORDS": (
            "ClickHouse column-oriented database OLAP analytics queries rows columns storage "
            "replication SQL performance approximate calculation"
        ),
        "MAX_SEGMENTS": "6",  # sections to turn into segments; more = longer, and slower to voice
        "USE_LLM": "false",  # "true" = Gemini writes and fact-checks (needs a key); "false" = offline, no model call
        "LLM_PROVIDER": "gemini",  # or "anthropic" for Claude
        "LLM_MODEL": "gemini-3.8-flash",  # used for every Gemini call: outline, script and fact-check
        "TTS_BACKEND": "chatterbox",  # "say" = macOS system voices, "placeholder" = fast tones for a test
        "SOURCE_ATTRIBUTION": (
            "Source material: 'What is ClickHouse?', ClickHouse Documentation "
            "(https://clickhouse.com/docs/get-started/about/intro). The documentation repository is "
            "licensed CC BY-NC-SA 4.0; this episode is a non-commercial adaptation. Confirm the "
            "current terms before publishing."
        ),
    },
    "samples": {  # the six bundled demo texts, about retrieval-augmented generation
        "SOURCE_URLS": "",
        "TOPIC": "How retrieval-augmented generation keeps AI answers grounded in sources",
        "TOPIC_KEYWORDS": (
            "retrieval augmented generation RAG language model documents passages embeddings vector "
            "search citations sources claims grounding hallucination fact-checking"
        ),
    },
}
# ================================================================================================
# Environment variables named PODCAST_NB_<NAME> win over everything here (the tests use them).
# The preset is read fresh on every run, so changing PRESET and re-running takes effect at once.
PRESET_VALUES = dict(PRESETS[os.environ.get("PODCAST_NB_PRESET", PRESET)])


def check_setup_is_current() -> None:
    """Replaced by the real check at the end of the setup cell, if (and only if) that cell finishes.

    Running this cell again puts this version back, so editing 'Your run' always forces the setup cell to run
    again, and a setup cell that failed halfway cannot be skipped over by running later cells.
    """
    raise RuntimeError(
        "'Your run' was edited, or the setup cell below it has not finished successfully. "
        "Fix what it reported, then Restart the kernel and use Run All."
    )

In [31]:
import hashlib
import importlib.util
import json
import random
import re
import shutil
import subprocess
import time
import warnings
import wave
from collections import Counter
from collections.abc import Iterator
from contextlib import contextmanager
from dataclasses import dataclass, field
from datetime import UTC, datetime, timedelta
from pathlib import Path
from uuid import NAMESPACE_URL, UUID, uuid4, uuid5

import numpy as np
import yaml
from IPython.display import Audio, Markdown, display
from pydantic import BaseModel, Field


def find_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the AI-podcast-engine repository.")


def shown(path: Path) -> str:
    """A repo-relative path when the file is inside the repo, otherwise the absolute path."""
    try:
        return str(path.relative_to(ROOT))
    except ValueError:
        return str(path)


def switch(name: str, default: object) -> object:
    """Read PODCAST_NB_<NAME>, else the 'Your run' cell, else the default. true/false become booleans."""
    raw = os.environ.get(f"PODCAST_NB_{name}")
    if raw is None:
        raw = PRESET_VALUES.get(name)  # the 'Your run' cell
    if raw is None:
        return default
    if isinstance(raw, bool):  # a real True or False typed in the 'Your run' cell
        return raw
    text = str(raw)  # numbers typed there arrive as text too
    return {"true": True, "false": False}.get(text.lower(), text)


# ---- switches --------------------------------------------------------------------------------
USE_LLM = switch(
    "USE_LLM", "auto"
)  # "auto" = use the LLM when a key is found; True / False force it
TTS_BACKEND = switch("TTS_BACKEND", "auto")  # "auto" | "say" | "chatterbox" | "placeholder"
USE_BGE = switch("USE_BGE", False)  # True downloads and runs the BGE embedding model
# Web pages to turn into an episode (comma-separated). Each section of a page becomes its own source.
SOURCE_URLS = [u.strip() for u in str(switch("SOURCE_URLS", "")).split(",") if u.strip()]
TOPIC = str(switch("TOPIC", ""))
if not TOPIC:
    raise ValueError("No TOPIC: set one in the 'Your run' cell (inside the preset you chose).")
# A topic profile is richer than the title: the title plus keywords the creator supplies.
TOPIC_KEYWORDS = str(switch("TOPIC_KEYWORDS", ""))
SEGMENT_ORDER = str(
    switch("SEGMENT_ORDER", "source" if SOURCE_URLS else "rank")
)  # or "rank" / "source"
# Below this a chunk is off-topic. Pages you name yourself are on-topic by choice, so no filtering.
TOPIC_THRESHOLD = float(str(switch("TOPIC_THRESHOLD", 0.0 if SOURCE_URLS else 0.07)))
FACTS_PER_SEGMENT = int(
    str(switch("FACTS_PER_SEGMENT", 8))
)  # caps how many sentences one segment quotes
URL_RELIABILITY = str(
    switch("URL_RELIABILITY", "")
)  # blank: look the domain up in source_tiers.yaml
SOURCE_ATTRIBUTION = str(
    switch("SOURCE_ATTRIBUTION", "")
)  # credit and licence line for the show notes
LIVE_SOURCES = switch("LIVE_SOURCES", False)  # True adds items fetched from the RSS feeds below
INJECT_HALLUCINATION = switch("INJECT_HALLUCINATION", not SOURCE_URLS)  # offline sample demo only
CLEAR_TTS_CACHE = switch("CLEAR_TTS_CACHE", True)  # True: the first build starts with a cold cache
PUSH_TO_S3 = switch("PUSH_TO_S3", False)  # True: upload the feed where the API serves it
RANKING_PROFILE = str(
    switch("RANKING_PROFILE", "explainer")
)  # "daily" favours fresh items; "explainer" favours authoritative ones
SEED = 7
LIVE_FEEDS = ["https://export.arxiv.org/rss/cs.CL"]
# Voices for the `say` backend (stock system voices). The `chatterbox` backend takes its voices
# from config/voices.yaml, where any cloned voice needs a consent record.
HOSTS = {"Alex": "Samantha", "Sam": "Daniel"}

# ---- environment ------------------------------------------------------------------------------
ROOT = find_root(Path.cwd().resolve())
NOTEBOOK_DIR = ROOT / "notebooks"
OUTPUT = Path(str(switch("OUTPUT_DIR", NOTEBOOK_DIR / "output")))
OUTPUT.mkdir(parents=True, exist_ok=True)
NOW = datetime.now(UTC)
RUN_ID = NOW.strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = OUTPUT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)
TTS_CACHE = OUTPUT / "tts_cache"
if CLEAR_TTS_CACHE:
    shutil.rmtree(TTS_CACHE, ignore_errors=True)
TTS_CACHE.mkdir(parents=True, exist_ok=True)
PROMPT_VERSIONS = {"outline": 1, "script": 2, "factcheck": 1}  # prompts/<name>.v<N>.md

# ---- the project's own code -------------------------------------------------------------------
from podcast_engine.common.config import get_settings
from podcast_engine.common.models import (
    Claim,
    ExpertiseLevel,
    FactCheckReport,
    ReliabilityTier,
    ScriptLine,
    Segment,
    ShowConfig,
    SourceItem,
    Tone,
)
from podcast_engine.common.queue import Stage, next_stage
from podcast_engine.common.storage import feed_key
from podcast_engine.stages.assemble.service import (
    MIN_SAMPLE_RATE_HZ,
    TARGET_LUFS,
    needs_normalisation,
)
from podcast_engine.stages.factcheck.service import route_after_factcheck
from podcast_engine.stages.index.service import chunk_words
from podcast_engine.stages.ingest.extract import (
    extract_page,
    fetch_html,
    source_tier,
    speakable_sentences,
)
from podcast_engine.stages.ingest.service import content_hash, has_changed
from podcast_engine.stages.publish.service import AI_DISCLOSURE, with_disclosure
from podcast_engine.stages.retrieve_plan.service import (
    combined_score,
    recency_score,
)
from podcast_engine.stages.script.service import load_prompt
from podcast_engine.stages.voice.service import segment_cache_key

settings = get_settings()
MAX_REWRITES = settings.max_rewrites
LLM_PROVIDER = str(switch("LLM_PROVIDER", settings.llm_provider)).lower()  # "gemini" or "anthropic"
if LLM_PROVIDER not in {"gemini", "anthropic"}:
    raise ValueError(f"LLM_PROVIDER must be gemini or anthropic, not {LLM_PROVIDER!r}")
DEFAULT_MODELS = {"gemini": "gemini-3.8-flash", "anthropic": "claude-sonnet-5-5"}
# A model you set yourself wins; otherwise the provider's default.
MODEL = str(switch("LLM_MODEL", os.environ.get("LLM_MODEL") or DEFAULT_MODELS[LLM_PROVIDER]))
KEY_VARIABLES = {
    "gemini": ("GEMINI_API_KEY", "GOOGLE_API_KEY"),
    "anthropic": ("ANTHROPIC_API_KEY",),
}


def read_api_key() -> str:
    """The key from the provider's environment variable, else from secrets/llm_api_key.txt."""
    for name in KEY_VARIABLES[LLM_PROVIDER]:
        if value := os.environ.get(name, "").strip():
            return value
    secret_file = ROOT / "secrets" / "llm_api_key.txt"
    return secret_file.read_text(encoding="utf-8").strip() if secret_file.exists() else ""


API_KEY = read_api_key()
LLM_ON = USE_LLM is True or (USE_LLM == "auto" and bool(API_KEY))
TTS_CHOICE = (
    ("say" if shutil.which("say") else "placeholder") if TTS_BACKEND == "auto" else str(TTS_BACKEND)
)
if TTS_CHOICE not in {"say", "chatterbox", "placeholder"}:
    raise ValueError(
        f"TTS_BACKEND must be auto, say, chatterbox or placeholder, not {TTS_BACKEND!r}"
    )
if TTS_CHOICE == "say" and shutil.which("say") is None:
    raise RuntimeError("TTS_BACKEND is 'say' but the macOS `say` command was not found.")
if TTS_CHOICE == "chatterbox" and importlib.util.find_spec("chatterbox") is None:
    raise RuntimeError(
        "TTS_BACKEND is 'chatterbox' but this kernel cannot import it. Select the `.venv-cloning` kernel "
        "(top right), or set TTS_BACKEND to 'say' or 'placeholder' in the 'Your run' cell."
    )
VOICES_FILE = Path(str(switch("VOICES_FILE", ROOT / "config" / "voices.yaml")))

KEY_HELP = {
    "gemini": (
        "Create a key at https://aistudio.google.com/apikey and put it in secrets/llm_api_key.txt "
        "(one line), or set GEMINI_API_KEY before starting the kernel."
    ),
    "anthropic": "Put a valid key in secrets/llm_api_key.txt, or set ANTHROPIC_API_KEY before starting the kernel.",
}


class LlmClient:
    """Common shape of the model clients: structured output in, validated object out, usage counted."""

    provider = ""

    def __init__(self, model: str) -> None:
        self.model = model
        self.usage: Counter[str] = Counter()
        self.models_used: Counter[str] = (
            Counter()
        )  # which model actually answered, for the manifest
        self.calls = 0

    def _record(self, tokens_in: int, tokens_out: int) -> None:
        self.calls += 1
        self.usage["input_tokens"] += tokens_in
        self.usage["output_tokens"] += tokens_out

    def check(self) -> None:
        raise NotImplementedError

    def parse(
        self, *, system: str, user: str, output_format: type[BaseModel], max_tokens: int = 16000
    ):
        raise NotImplementedError


class GeminiClient(LlmClient):
    """Google Gemini through the `google-genai` SDK, asking for JSON that matches a Pydantic schema."""

    provider = "gemini"
    RETRIES = 8  # waits of about 2, 4, 8, 16, 32, 32 and 32 seconds: about two minutes per call
    RETRIES_BEFORE_FALLBACK = (
        3  # with a fallback available: waits of about 2 and 4 seconds, then switch
    )

    def __init__(self, api_key: str, model: str, fallback: str = "") -> None:
        super().__init__(model)
        self.fallback = fallback if fallback != model else ""
        self.active = (
            model  # the model used for new calls; becomes the fallback once the main one stays busy
        )
        self.busy_retries = 0
        from google import genai

        try:  # with no key given the SDK reads GEMINI_API_KEY or GOOGLE_API_KEY
            self.client = genai.Client(api_key=api_key) if api_key else genai.Client()
        except ValueError as error:
            raise RuntimeError(f"No Gemini credentials were found. {KEY_HELP['gemini']}") from error

    def check(self) -> None:
        """Fail now, with a clear message, rather than minutes into a run. Looking up a model costs nothing."""
        from google.genai import errors

        try:
            self.client.models.get(model=self.model)
        except errors.ClientError as error:
            if error.code == 404:
                raise RuntimeError(
                    f"Model {self.model!r} was not found for this key. Set LLM_MODEL to one it can use."
                ) from error
            if error.code == 429:
                raise RuntimeError(
                    "Gemini says the quota or rate limit is used up. Wait, or enable billing for the key."
                ) from error
            why = getattr(error, "message", None) or str(error)[:140]
            raise RuntimeError(
                f"Gemini rejected the request: {why} {KEY_HELP['gemini']}"
            ) from error
        except errors.ServerError as error:
            raise RuntimeError(f"Gemini is unavailable right now: {str(error)[:140]}") from error

    def _call(self, model: str, user: str, config, tries: int):
        """One request, patiently retried: 503 "high demand" spikes and 429 rate limits are usually brief."""
        from google.genai import errors

        for attempt in range(tries):
            try:
                return self.client.models.generate_content(
                    model=model, contents=user, config=config
                )
            except (errors.ServerError, errors.ClientError) as error:
                busy = isinstance(error, errors.ServerError) or error.code == 429
                if not busy or attempt == tries - 1:
                    raise
                self.busy_retries += 1
                wait = min(2 ** (attempt + 1), 32) + random.random()
                print(f"  {model} is busy ({error.code}); retrying in {wait:.0f} s", flush=True)
                time.sleep(wait)

    def parse(
        self, *, system: str, user: str, output_format: type[BaseModel], max_tokens: int = 16000
    ):
        from google.genai import errors, types
        from pydantic import ValidationError

        schema = output_format.model_json_schema()
        with_schema = types.GenerateContentConfig(
            system_instruction=system,
            response_mime_type="application/json",
            response_json_schema=schema,
            max_output_tokens=max_tokens,
            seed=SEED,  # makes a rerun as repeatable as the API allows
        )
        plain_json = types.GenerateContentConfig(  # used only if the API refuses the schema
            system_instruction=f"{system}\n\nReply with JSON matching this JSON Schema:\n{json.dumps(schema)}",
            response_mime_type="application/json",
            max_output_tokens=max_tokens,
            seed=SEED,
        )
        candidates = [
            self.active,
            *([self.fallback] if self.fallback and self.fallback != self.active else []),
        ]
        for position, model in enumerate(candidates):
            has_next = position < len(candidates) - 1
            tries = self.RETRIES_BEFORE_FALLBACK if has_next else self.RETRIES
            try:
                try:
                    response = self._call(model, user, with_schema, tries)
                except errors.ClientError as error:
                    if error.code != 400 or "schema" not in str(error).lower():
                        raise
                    response = self._call(model, user, plain_json, tries)
                break
            except (errors.ServerError, errors.ClientError) as error:
                still_busy = isinstance(error, errors.ServerError) or error.code == 429
                if not still_busy or not has_next:
                    raise
                self.active = candidates[
                    position + 1
                ]  # remember: do not wait on the busy model again
                print(
                    f"  {model} stayed busy; using {self.active} for the rest of this run",
                    flush=True,
                )
        self.models_used[model] += 1
        meta = response.usage_metadata
        # Thinking tokens are billed as output.
        self._record(
            (meta.prompt_token_count or 0) if meta else 0,
            ((meta.candidates_token_count or 0) + (meta.thoughts_token_count or 0)) if meta else 0,
        )
        candidate = response.candidates[0] if response.candidates else None
        reason = (
            candidate.finish_reason.name
            if candidate and candidate.finish_reason
            else "NO_CANDIDATE"
        )
        if reason in {"SAFETY", "PROHIBITED_CONTENT", "BLOCKLIST", "SPII", "RECITATION"}:
            raise RuntimeError(f"Gemini declined this request (finish_reason={reason}).")
        if reason == "MAX_TOKENS":
            raise RuntimeError(
                f"Gemini's answer was cut off at {max_tokens} tokens; raise max_tokens."
            )
        text = response.text
        if not text:
            raise RuntimeError(
                f"Gemini returned no text (finish_reason={reason}, feedback={response.prompt_feedback})."
            )
        try:
            return output_format.model_validate_json(text)
        except ValidationError as error:
            raise RuntimeError(
                f"Gemini's JSON does not match {output_format.__name__}: {error}\n{text[:300]}"
            ) from error


class ClaudeClient(LlmClient):
    """Anthropic Claude through the `anthropic` SDK, using structured outputs."""

    provider = "anthropic"

    def __init__(self, api_key: str, model: str) -> None:
        super().__init__(model)
        import anthropic

        # With no key given the SDK looks for ANTHROPIC_API_KEY, ANTHROPIC_AUTH_TOKEN or an `ant auth login`.
        self.client = anthropic.Anthropic(api_key=api_key) if api_key else anthropic.Anthropic()

    def check(self) -> None:
        import anthropic

        try:
            self.client.models.retrieve(self.model)
        except TypeError as error:  # the SDK found no credentials at all
            raise RuntimeError(
                f"No Anthropic credentials were found. {KEY_HELP['anthropic']}"
            ) from error
        except anthropic.AuthenticationError as error:
            raise RuntimeError(f"Anthropic rejected the key. {KEY_HELP['anthropic']}") from error
        except anthropic.NotFoundError as error:
            raise RuntimeError(
                f"Model {self.model!r} was not found for this key. Set LLM_MODEL to one it can use."
            ) from error
        except anthropic.APIConnectionError as error:
            raise RuntimeError(f"Could not reach the Anthropic API: {error}") from error

    def parse(
        self, *, system: str, user: str, output_format: type[BaseModel], max_tokens: int = 16000
    ):
        response = self.client.messages.parse(
            model=self.model,
            max_tokens=max_tokens,
            system=system,
            messages=[{"role": "user", "content": user}],
            output_format=output_format,
        )
        self._record(response.usage.input_tokens, response.usage.output_tokens)
        self.models_used[self.model] += 1
        if response.stop_reason == "refusal":
            raise RuntimeError(f"The model declined this request: {response.stop_details}")
        if response.parsed_output is None:
            raise RuntimeError(
                f"No structured output returned (stop_reason={response.stop_reason})"
            )
        return response.parsed_output


LLM: LlmClient | None = None
if LLM_ON and LLM_PROVIDER == "gemini":
    # Every Gemini call uses MODEL. Another model answers only if you ask for one with LLM_FALLBACK_MODEL.
    fallback = str(switch("LLM_FALLBACK_MODEL", ""))
    LLM = GeminiClient(API_KEY, MODEL, "" if fallback.lower() == "none" else fallback)
elif LLM_ON:
    LLM = ClaudeClient(API_KEY, MODEL)
if LLM is not None:
    LLM.check()

SR = 24_000  # PRD: 24 kHz or higher
rng = random.Random(SEED)


def show_table(headers: list[str], rows: list[list[object]]) -> None:
    head = "| " + " | ".join(headers) + " |\n|" + "---|" * len(headers) + "\n"
    body = "\n".join("| " + " | ".join(str(cell) for cell in row) + " |" for row in rows)
    display(Markdown(head + body))


show_table(
    ["Capability", "Using"],
    [
        [
            "Script / outline / fact-check",
            f"{LLM_PROVIDER} `{MODEL}`" if LLM_ON else f"offline stand-in (USE_LLM is {USE_LLM!r})",
        ],
        ["Embeddings", settings.embedding_model if USE_BGE else "hashed bag-of-words (offline)"],
        [
            "Voices",
            {
                "say": "macOS `say` system voices",
                "chatterbox": f"Chatterbox, voices from `{shown(VOICES_FILE)}`",
                "placeholder": "placeholder tones (not speech)",
            }[TTS_CHOICE],
        ],
        [
            "Source",
            "\n".join(SOURCE_URLS) if SOURCE_URLS else "the bundled sample texts (no web page)",
        ],
        ["Preset", f"`{os.environ.get('PODCAST_NB_PRESET', PRESET)}`"],
        ["Topic", TOPIC],
        ["Run folder", f"`{shown(RUN_DIR)}`"],
    ],
)


# Names whose values the rest of the notebook has already acted on by the end of this cell.
SETTINGS_WATCHED = (
    "USE_LLM",
    "LLM_PROVIDER",
    "LLM_MODEL",
    "TTS_BACKEND",
    "SOURCE_URLS",
    "TOPIC",
    "MAX_SEGMENTS",
)


def current_settings() -> dict[str, str]:
    return {name: str(switch(name, None)) for name in SETTINGS_WATCHED}


SETUP_SETTINGS = current_settings()


def check_setup_is_current() -> None:  # noqa: F811 - deliberately replaces the placeholder in 'Your run'
    """Stop, loudly, if 'Your run' was edited after this setup cell ran.

    The setup cell has already created the model client, chosen the voices and read the sources. Re-running
    only some cells would carry on with the old values (for example, still calling Gemini after USE_LLM was
    set to false), so the notebook refuses to continue until you re-run from the top.
    """
    now = current_settings()
    changed = {
        name: (SETUP_SETTINGS[name], now[name]) for name in now if now[name] != SETUP_SETTINGS[name]
    }
    if changed:
        details = "; ".join(f"{name}: {old} -> {new}" for name, (old, new) in changed.items())
        raise RuntimeError(
            "'Your run' was changed after the setup cell ran, so the rest of the notebook is still using the "
            f"old settings ({details}). Restart the kernel and use Run All."
        )

RuntimeError: No Gemini credentials were found. Create a key at https://aistudio.google.com/apikey and put it in secrets/llm_api_key.txt (one line), or set GEMINI_API_KEY before starting the kernel.

## 1. The show (FR1)

A creator defines a show: topic, tone, length, number of hosts, language and expertise level. The
project's own `ShowConfig` model validates it, exactly as the API does.

In [ ]:
def load_url_docs(urls: list[str]) -> list[dict]:
    """Fetch pages and return one document per section, so each can change and be cited on its own."""
    tiers = yaml.safe_load((ROOT / "config" / "source_tiers.yaml").read_text())
    docs: list[dict] = []
    for url in urls:
        page = extract_page(fetch_html(url))
        if not page.sections:
            raise RuntimeError(f"No usable text found at {url}")
        age = max((NOW - page.published_at).days, 0) if page.published_at else 0
        tier = URL_RELIABILITY or source_tier(url, tiers).value
        for section in page.sections:
            anchored = f"{url}#{section.anchor}" if section.anchor else url
            docs.append(
                {
                    "url": anchored,
                    "title": section.heading,
                    "page_title": page.title,
                    "age_days": age,
                    "reliability": tier,
                    "text": section.text,
                }
            )
        print(f"{url}: {len(page.sections)} sections ({page.title})")
        print(f"  not used, too little speakable text: {list(page.dropped) or 'none'}")
    return docs


if SOURCE_URLS:
    RAW_DOCS: list[dict] = load_url_docs(SOURCE_URLS)
else:
    RAW_DOCS = json.loads((NOTEBOOK_DIR / "sample_sources.json").read_text())["documents"]

SHOW = ShowConfig(
    show_id=uuid5(
        NAMESPACE_URL, SOURCE_URLS[0] if SOURCE_URLS else "bundled-samples"
    ),  # only an id; nothing is fetched
    topic=TOPIC,
    tone=Tone.CASUAL,
    target_minutes=3,  # the PRD target is 7; the sample sources only hold enough material for ~3
    host_count=2,
    language="en",
    expertise=ExpertiseLevel.BEGINNER,
    sources=tuple(doc["url"] for doc in RAW_DOCS),
)
HOST_NAMES = list(HOSTS)[: SHOW.host_count]
EPISODE_ID = uuid4()
display(
    Markdown(
        f"**Topic** {SHOW.topic}  \n**Source** {', '.join(SOURCE_URLS) or 'bundled sample texts'}  \n"
        f"**Hosts** {HOST_NAMES}  \n**Show** `{SHOW.show_id}`  \n**Episode** `{EPISODE_ID}`"
    )
)
SHOW

## 2. Stage 1: Ingest (FR2)

> *1 Ingest: RSS, URLs, PDFs, search; detect changes by hash.*

Each source gets a content hash. Running ingest twice shows **change detection**: the second run finds
nothing new, so downstream stages would not run at all.

A web page is split at its headings and **each section is its own source**, with its own hash and its own
anchor, so a change on the page re-runs only the segment built from that section, and show-notes citations link
to `page#section`. Sections with too little speakable text are skipped, and the cell prints which ones. The
refresh step fetches the page again: if no section changed, nothing re-runs. RSS summaries and the bundled
sample texts work the same way. PDF and search-result ingestion are not covered.

In [ ]:
check_setup_is_current()


@dataclass
class SourceStore:
    items: dict[UUID, SourceItem] = field(default_factory=dict)
    texts: dict[UUID, str] = field(default_factory=dict)


STORE = SourceStore()
TIMINGS: dict[str, list[float]] = {}
ROUTE_LOG: list[str] = []


@contextmanager
def timed(stage: str) -> Iterator[None]:
    """Collect wall-clock time per stage (the PRD's 'pipeline timing logs')."""
    started = time.perf_counter()
    try:
        yield
    finally:
        TIMINGS.setdefault(stage, []).append(time.perf_counter() - started)


def fetch_live_feed(url: str, limit: int = 3) -> list[dict]:
    """Summaries from an RSS feed. Needs network; returns [] and says why if it fails."""
    try:
        import feedparser
        import httpx

        reply = httpx.get(url, timeout=20, follow_redirects=True)  # httpx brings its own CA bundle
        reply.raise_for_status()
        parsed = feedparser.parse(reply.content)
        if parsed.bozo and not parsed.entries:
            raise ValueError(f"not a readable feed: {parsed.bozo_exception}")
        docs = []
        for entry in parsed.entries[:limit]:
            summary = re.sub(r"<[^>]+>", " ", entry.get("summary", ""))
            summary = " ".join(summary.split())
            published = entry.get("published_parsed")
            age = (NOW - datetime(*published[:6], tzinfo=UTC)).days if published else 1
            if summary:
                docs.append(
                    {
                        "url": entry.get("link", url),
                        "title": entry.get("title", "Untitled"),
                        "age_days": max(age, 0),
                        "reliability": "primary",
                        "text": summary,
                    }
                )
        return docs
    except Exception as error:  # network or parse problems should not stop the notebook
        print(f"live feed skipped ({url}): {error}")
        return []


def ingest(docs: list[dict], store: SourceStore, *, version: int = 1) -> list[UUID]:
    """Store new or changed documents and return the ids that changed."""
    changed: list[UUID] = []
    for doc in docs:
        text = doc["text_v2"] if version == 2 and "text_v2" in doc else doc["text"]
        item_id = uuid5(NAMESPACE_URL, doc["url"])
        previous = store.items.get(item_id)
        if not has_changed(previous.content_hash if previous else None, text):
            continue
        age = 0 if version == 2 and "text_v2" in doc else doc["age_days"]
        store.items[item_id] = SourceItem(
            item_id=item_id,
            show_id=SHOW.show_id,
            url=doc["url"],
            title=doc["title"],
            published_at=NOW - timedelta(days=age),
            content_hash=content_hash(text),
            reliability=ReliabilityTier(doc["reliability"]),
        )
        store.texts[item_id] = text
        changed.append(item_id)
    return changed


if LIVE_SOURCES:
    for feed_url in LIVE_FEEDS:
        RAW_DOCS += fetch_live_feed(feed_url)

with timed("1 ingest"):
    first_run = ingest(RAW_DOCS, STORE)
    second_run = ingest(RAW_DOCS, STORE)  # nothing changed, so nothing should be returned

show_table(
    ["Item", "Reliability", "Age (days)", "Hash"],
    [
        [
            i.title,
            i.reliability.value,
            (NOW - i.published_at).days if i.published_at else "?",
            i.content_hash[:10],
        ]
        for i in STORE.items.values()
    ],
)
print(f"first run changed {len(first_run)} items; second run changed {len(second_run)}")
assert len(first_run) == len(RAW_DOCS)
assert second_run == []
ROUTE_LOG.append(f"{Stage.INGEST.value} -> {next_stage(Stage.INGEST).value}")

## 3. Stage 2: Index (FR3)

> *2 Index: clean, chunk, embed; vector store plus metadata.*

Documents are split with the project's `chunk_words` (overlapping windows), embedded, and kept in an
in-memory vector store. The repository stores the same data in Postgres/pgvector (`db/init/002_schema.sql`).
Embeddings are **cached by text**, which is the PRD's cost saving: unchanged chunks are never re-embedded.

In [ ]:
STOPWORDS = set(
    "a an and are as at be been but by can for from has have how in is it its not of on or so that the their "
    "them then there these they this to was were what when which who will with you your into than also such "
    "often most more many some other each every about because instead while only still even".split()
)


def stem(token: str) -> str:
    for suffix in ("ing", "ed", "es", "s"):
        if token.endswith(suffix) and len(token) - len(suffix) >= 4:
            return token[: -len(suffix)]
    return token


def content_tokens(text: str) -> list[str]:
    return [
        stem(t) for t in re.findall(r"[a-z0-9]+", text.lower()) if t not in STOPWORDS and len(t) > 2
    ]


class HashingEmbedder:
    """Offline embedder: signed hashing of stemmed words. Deterministic and dependency-free."""

    name = "hashing-bow-512"
    dim = 512

    def encode(self, texts: list[str]) -> np.ndarray:
        out = np.zeros((len(texts), self.dim), dtype=np.float32)
        for row, text in enumerate(texts):
            for token in content_tokens(text):
                digest = hashlib.md5(token.encode(), usedforsecurity=False).digest()
                slot = int.from_bytes(digest[:4], "big") % self.dim
                out[row, slot] += 1.0 if digest[4] % 2 == 0 else -1.0
            norm = np.linalg.norm(out[row])
            if norm:
                out[row] /= norm
        return out


class SentenceTransformerEmbedder:
    def __init__(self, model_name: str) -> None:
        from sentence_transformers import SentenceTransformer

        self.name = model_name
        self._model = SentenceTransformer(model_name)
        self.dim = int(self._model.get_sentence_embedding_dimension())

    def encode(self, texts: list[str]) -> np.ndarray:
        return np.asarray(self._model.encode(texts, normalize_embeddings=True), dtype=np.float32)


class CachedEmbedder:
    """Embedding cache keyed by text hash; counts hits so the saving is visible."""

    def __init__(self, inner: HashingEmbedder | SentenceTransformerEmbedder) -> None:
        self.inner = inner
        self.name = inner.name
        self._cache: dict[str, np.ndarray] = {}
        self.hits = 0
        self.misses = 0

    def encode(self, texts: list[str]) -> np.ndarray:
        keys = [content_hash(t) for t in texts]
        missing = [(k, t) for k, t in zip(keys, texts, strict=True) if k not in self._cache]
        if missing:
            vectors = self.inner.encode([t for _, t in missing])
            for (key, _), vector in zip(missing, vectors, strict=True):
                self._cache[key] = vector
        self.misses += len(missing)
        self.hits += len(texts) - len(missing)
        return np.stack([self._cache[k] for k in keys])


EMBEDDER = CachedEmbedder(
    SentenceTransformerEmbedder(settings.embedding_model) if USE_BGE else HashingEmbedder()
)


@dataclass(eq=False)
class Chunk:
    chunk_id: UUID
    item_id: UUID
    position: int
    text: str
    vector: np.ndarray


class VectorStore:
    def __init__(self) -> None:
        self.chunks: dict[UUID, Chunk] = {}

    def replace_item(self, item_id: UUID, chunks: list[Chunk]) -> None:
        self.chunks = {cid: c for cid, c in self.chunks.items() if c.item_id != item_id}
        self.chunks.update({c.chunk_id: c for c in chunks})

    def of_item(self, item_id: UUID) -> list[Chunk]:
        return sorted(
            (c for c in self.chunks.values() if c.item_id == item_id), key=lambda c: c.position
        )


INDEX = VectorStore()
# The overlap must be at least as long as the longest sentence, so every sentence sits whole inside
# at least one chunk (the offline writer quotes whole sentences and cites the chunk they came from).
CHUNK_WORDS, CHUNK_OVERLAP = 80, 40


def index_items(item_ids: list[UUID]) -> int:
    """Chunk and embed the given items, replacing any older chunks of the same item."""
    added = 0
    for item_id in item_ids:
        pieces = chunk_words(STORE.texts[item_id], max_words=CHUNK_WORDS, overlap=CHUNK_OVERLAP)
        vectors = EMBEDDER.encode(pieces)
        INDEX.replace_item(
            item_id,
            [
                Chunk(uuid5(item_id, f"{n}:{content_hash(text)[:12]}"), item_id, n, text, vector)
                for n, (text, vector) in enumerate(zip(pieces, vectors, strict=True))
            ],
        )
        added += len(pieces)
    return added


with timed("2 index"):
    n_chunks = index_items(list(STORE.items))

ROUTE_LOG.append(f"{Stage.INDEX.value} -> {next_stage(Stage.INDEX).value}")
show_table(
    ["Item", "Chunks", "First chunk"],
    [
        [STORE.items[i].title, len(INDEX.of_item(i)), INDEX.of_item(i)[0].text[:70] + "..."]
        for i in STORE.items
    ],
)
print(
    f"{n_chunks} chunks, {EMBEDDER.name}, embedding cache: {EMBEDDER.misses} computed / {EMBEDDER.hits} reused"
)

## 4. Stage 3: Retrieve and plan (FR3, FR4)

> *3 Retrieve and plan: rank by relevance and date; build the segment outline.*

Three PRD ideas happen here:

1. **Topic matching:** chunks are scored against the show's topic; only those above a threshold enter
   the candidate pool. The off-topic sourdough text should be excluded.
2. **Freshness-aware retrieval:** ranking blends relevance with recency (the project's `recency_score`
   and `combined_score`). A *daily* show weights recency heavily; an *explainer* weights authority.
3. **Segment planning:** the outline is built from the pool, and each segment is tagged with the
   sources it uses. That tag is what makes incremental regeneration possible later.

In [ ]:
TOPIC_VECTOR = EMBEDDER.encode([f"{SHOW.topic}. {TOPIC_KEYWORDS}"])[0]
HALF_LIFE_DAYS = 14
AUTHORITY = {
    ReliabilityTier.PRIMARY: 1.0,
    ReliabilityTier.ESTABLISHED: 0.6,
    ReliabilityTier.UNVERIFIED: 0.2,
}
RANKING_PROFILES = {
    "daily": {"recency_weight": 0.8, "authority_weight": 0.0},
    "explainer": {"recency_weight": 0.1, "authority_weight": 0.4},
}
MAX_SEGMENTS = int(str(switch("MAX_SEGMENTS", 5)))  # fewer segments = a shorter, faster episode
PASSAGES_PER_SEGMENT = int(
    str(switch("PASSAGES_PER_SEGMENT", 6 if LLM_ON else 3))
)  # an LLM can use more


@dataclass
class Ranked:
    chunk: Chunk
    item: SourceItem
    relevance: float
    recency: float
    authority: float
    score: float


def rank_pool(profile: str, *, only_items: set[UUID] | None = None) -> list[Ranked]:
    """Topic-match every chunk, drop those below threshold, rank the rest."""
    weights = RANKING_PROFILES[profile]
    pool: list[Ranked] = []
    for chunk in INDEX.chunks.values():
        if only_items is not None and chunk.item_id not in only_items:
            continue
        relevance = float(chunk.vector @ TOPIC_VECTOR)
        if relevance < TOPIC_THRESHOLD:
            continue
        item = STORE.items[chunk.item_id]
        recency = recency_score(item.published_at, now=NOW, half_life_days=HALF_LIFE_DAYS)
        authority = AUTHORITY[item.reliability]
        blended = combined_score(relevance, recency, recency_weight=weights["recency_weight"])
        score = (1 - weights["authority_weight"]) * blended + weights[
            "authority_weight"
        ] * authority
        pool.append(Ranked(chunk, item, relevance, recency, authority, score))
    return sorted(pool, key=lambda r: r.score, reverse=True)


with timed("3 retrieve+plan"):
    all_scores = {
        item.title: float(max(c.vector @ TOPIC_VECTOR for c in INDEX.of_item(item.item_id)))
        for item in STORE.items.values()
    }
    pool = rank_pool(RANKING_PROFILE)

display(Markdown("**Topic matching** (best chunk per source against the show topic)"))
show_table(
    ["Source", "Best relevance", f"Threshold {TOPIC_THRESHOLD}"],
    [
        [t, f"{s:.2f}", "in pool" if s >= TOPIC_THRESHOLD else "**excluded**"]
        for t, s in all_scores.items()
    ],
)

display(Markdown("**Same pool, two ranking profiles**"))
for name in RANKING_PROFILES:
    show_table(
        ["Rank", "Source", "Relevance", "Recency", "Authority", "Score"],
        [
            [
                n + 1,
                r.item.title,
                f"{r.relevance:.2f}",
                f"{r.recency:.2f}",
                f"{r.authority:.1f}",
                f"{r.score:.3f}",
            ]
            for n, r in enumerate(rank_pool(name)[:4])
        ],
    )
    display(Markdown(f"*{name}*"))
assert "Sourdough starter basics (off-topic on purpose)" not in {r.item.title for r in pool}

In [ ]:
def passages_for(source_item_ids: tuple[UUID, ...]) -> list[Ranked]:
    """The best passages for one segment, drawn only from that segment's sources."""
    return rank_pool(RANKING_PROFILE, only_items=set(source_item_ids))[:PASSAGES_PER_SEGMENT]


class OutlineSegment(BaseModel):
    title: str
    passage_ids: list[str]


class Outline(BaseModel):
    segments: list[OutlineSegment]


def label_passages(pool_: list[Ranked]) -> dict[str, Ranked]:
    return {f"P{n + 1}": r for n, r in enumerate(pool_)}


def plan_outline_offline(pool_: list[Ranked]) -> list[Segment]:
    """One segment per source, best-ranked source first."""
    ordered: list[UUID] = []
    for ranked in pool_:
        if ranked.item.item_id not in ordered:
            ordered.append(ranked.item.item_id)
    ordered = ordered[:MAX_SEGMENTS]
    if SEGMENT_ORDER == "source":  # tell the page's story in the page's order
        page_order = {item_id: n for n, item_id in enumerate(STORE.items)}
        ordered.sort(key=page_order.__getitem__)
    return [
        Segment(
            segment_id=uuid5(EPISODE_ID, f"segment-{n}"),
            position=n,
            title=STORE.items[item_id].title,
            source_item_ids=(item_id,),
        )
        for n, item_id in enumerate(ordered, start=1)
    ]


def plan_outline_llm(pool_: list[Ranked]) -> list[Segment]:
    labelled = label_passages(pool_[:24])
    listing = "\n".join(
        f"[{pid}] ({r.item.title}, {r.item.published_at:%Y-%m-%d}) {r.chunk.text}"
        for pid, r in labelled.items()
    )
    outline = LLM.parse(
        system=load_prompt(ROOT / "prompts", "outline", PROMPT_VERSIONS["outline"]),
        user=f"Show topic: {SHOW.topic}\nUse at most {MAX_SEGMENTS} segments.\n\nPassages:\n{listing}",
        output_format=Outline,
    )
    segments = []
    for n, planned in enumerate(outline.segments[:MAX_SEGMENTS], start=1):
        items = tuple(
            dict.fromkeys(labelled[p].item.item_id for p in planned.passage_ids if p in labelled)
        )
        if not items:
            raise ValueError(
                f"segment '{planned.title}' cites no known passage: {planned.passage_ids}"
            )
        segments.append(
            Segment(
                segment_id=uuid5(EPISODE_ID, f"segment-{n}"),
                position=n,
                title=planned.title,
                source_item_ids=items,
            )
        )
    if (
        SEGMENT_ORDER == "source"
    ):  # tell the page's story in the page's order, whatever order the model chose
        page_order = {item_id: n for n, item_id in enumerate(STORE.items)}
        segments.sort(key=lambda seg: min(page_order[i] for i in seg.source_item_ids))
        segments = [
            seg.model_copy(update={"position": n, "segment_id": uuid5(EPISODE_ID, f"segment-{n}")})
            for n, seg in enumerate(segments, start=1)
        ]
    return segments

In [ ]:
check_setup_is_current()

with timed("3 retrieve+plan"):
    SEGMENTS = plan_outline_llm(pool) if LLM_ON else plan_outline_offline(pool)
    PASSAGES: dict[int, list[Ranked]] = {
        s.position: passages_for(s.source_item_ids) for s in SEGMENTS
    }

ROUTE_LOG.append(f"{Stage.RETRIEVE_PLAN.value} -> {next_stage(Stage.RETRIEVE_PLAN).value}")
show_table(
    ["#", "Segment", "Uses source(s)", "Passages"],
    [
        [
            s.position,
            s.title,
            ", ".join(STORE.items[i].url.rsplit("/", 1)[-1] for i in s.source_item_ids),
            len(PASSAGES[s.position]),
        ]
        for s in SEGMENTS
    ],
)

## 5. Stage 4: Script (FR4, FR5)

> *4 Script: dialogue with citations from retrieved passages.*

The script is written **one segment at a time**, so a changed source can later re-script just its
segment. Each spoken line is either

* a **claim line**: one factual statement that must cite at least one passage (FR5), or
* a **banter line**: framing and reactions with no facts and no citation.

Prompts are versioned files in `prompts/` (`script.v2.md` for the script) and the versions are logged in the run
manifest, so an experiment can be rerun.

**With an LLM (the default),** the model writes each segment in its own words from that segment's passages,
marking each factual line as a claim and listing the passages that support it. The prompt forbids copying
sentences and adding facts the passages do not contain. **Offline,** the writer is *extractive*: it quotes whole
sentences from the passages, so the first draft is supported by construction. To exercise the gate on the
bundled samples, `INJECT_HALLUCINATION` plants two bad claims in segment 2 of the **first draft only** (one cited
to a passage that does not say it, one with no citation at all). It is off for web pages, where nothing is
planted.

In [ ]:
check_setup_is_current()


class DraftLine(BaseModel):
    speaker: str
    text: str
    is_claim: bool = Field(
        description="True for a factual statement, False for framing or reactions"
    )
    passage_ids: list[str] = Field(
        default_factory=list, description="Passage ids that support a claim"
    )


class DraftSegment(BaseModel):
    lines: list[DraftLine]


OPEN_LINES = ["Let's start with {title}.", "Next up: {title}.", "Now, {title}."]
QUESTION_LINES = [
    "Here is the question: {title}?",
    "Let's tackle this one: {title}?",
    "So, {title}?",
]


def opening_line(title: str) -> str:
    """Introduce a segment by its heading, keeping names like ClickHouse in their own case."""
    plain = title.rstrip("?.! ")
    if title.rstrip().endswith("?"):
        return rng.choice(QUESTION_LINES).format(title=plain)
    lowered = plain[0].lower() + plain[1:] if plain[1:2].islower() else plain
    return rng.choice(OPEN_LINES).format(title=lowered)


REACT_LINES = [
    "That makes sense.",
    "Right, that is the key idea.",
    "Good point, say more.",
    "I like that.",
]
PLANTED_CLAIMS = [
    # cites a real passage that does not say this (misattribution)
    ("The first system of this kind was built in 1952 by a team of librarians in Vienna.", True),
    # no citation at all
    ("Every modern system checks its answers against an encyclopedia before replying.", False),
]


def to_script_lines(
    lines: list[DraftLine], labelled: dict[str, Ranked], position: int
) -> list[ScriptLine]:
    out = []
    for line in lines:
        if line.speaker not in HOST_NAMES:
            raise ValueError(f"unknown speaker {line.speaker!r}; hosts are {HOST_NAMES}")
        claims: tuple[Claim, ...] = ()
        if line.is_claim:
            chunk_ids = tuple(labelled[p].chunk.chunk_id for p in line.passage_ids if p in labelled)
            claims = (
                Claim(
                    claim_id=uuid5(EPISODE_ID, f"{position}/{line.text}"),
                    text=line.text,
                    chunk_ids=chunk_ids,
                ),
            )
        out.append(ScriptLine(speaker=line.speaker, text=line.text, claims=claims))
    return out


def write_segment_offline(
    segment: Segment, passages: list[Ranked], *, attempt: int, avoid: set[str]
) -> list[ScriptLine]:
    labelled = label_passages(passages)
    pid_of = {r.chunk.chunk_id: pid for pid, r in labelled.items()}
    lines = [DraftLine(speaker=HOST_NAMES[0], text=opening_line(segment.title), is_claim=False)]
    # Whole sentences only: a sentence is quoted if it sits inside one passage, and that passage is cited.
    found: dict[tuple[UUID, int], tuple[str, UUID]] = {}
    for r in passages:
        for index, sentence in enumerate(speakable_sentences(STORE.texts[r.chunk.item_id])):
            if sentence in r.chunk.text:
                found.setdefault((r.chunk.item_id, index), (sentence, r.chunk.chunk_id))
    facts = [found[key] for key in sorted(found, key=lambda k: k[1])][:FACTS_PER_SEGMENT]
    for n, (sentence, chunk_id) in enumerate(facts):
        speaker = HOST_NAMES[(n + 1) % len(HOST_NAMES)]
        lines.append(
            DraftLine(speaker=speaker, text=sentence, is_claim=True, passage_ids=[pid_of[chunk_id]])
        )
        if n % 2 == 1:
            reactor = HOST_NAMES[n % len(HOST_NAMES)]
            lines.append(DraftLine(speaker=reactor, text=rng.choice(REACT_LINES), is_claim=False))
    if INJECT_HALLUCINATION and attempt == 0 and segment.position == 2:
        for text, cites in PLANTED_CLAIMS:
            lines.append(
                DraftLine(
                    speaker=HOST_NAMES[0],
                    text=text,
                    is_claim=True,
                    passage_ids=["P1"] if cites else [],
                )
            )
    lines = [ln for ln in lines if ln.text not in avoid]
    return to_script_lines(lines, labelled, segment.position)


def write_segment_llm(
    segment: Segment, passages: list[Ranked], *, attempt: int, avoid: set[str]
) -> list[ScriptLine]:
    labelled = label_passages(passages)
    words = max(60, SHOW.target_minutes * 150 // max(len(SEGMENTS), 1))
    style = (
        f"Tone: {SHOW.tone.value}. Audience expertise: {SHOW.expertise.value}. Language: {SHOW.language}. "
        f"Hosts: {', '.join(HOST_NAMES)}. About {words} spoken words for this segment. "
        "Every line with a factual statement must set is_claim=true and list the passage ids that support it; "
        "framing and reactions set is_claim=false with no passage ids."
    )
    listing = "\n".join(f"[{pid}] ({r.item.title}) {r.chunk.text}" for pid, r in labelled.items())
    redo = ""
    if avoid:
        redo = (
            "\n\nA reviewer found these statements unsupported by the passages. Do not repeat them:\n- "
            + "\n- ".join(sorted(avoid))
        )
    draft = LLM.parse(
        system=load_prompt(ROOT / "prompts", "script", PROMPT_VERSIONS["script"]) + "\n\n" + style,
        user=(
            f"Show: {SHOW.topic}\nThis is segment {segment.position} of {len(SEGMENTS)}.\n\n"
            f"Segment title: {segment.title}\n\nPassages:\n{listing}{redo}"
        ),
        output_format=DraftSegment,
    )
    return to_script_lines(draft.lines, labelled, segment.position)


def write_segments(
    positions: list[int], *, attempt: int, avoid: set[str]
) -> dict[int, list[ScriptLine]]:
    writer = write_segment_llm if LLM_ON else write_segment_offline
    by_position = {s.position: s for s in SEGMENTS}
    return {p: writer(by_position[p], PASSAGES[p], attempt=attempt, avoid=avoid) for p in positions}


with timed("4 script"):
    SCRIPT = write_segments([s.position for s in SEGMENTS], attempt=0, avoid=set())

ROUTE_LOG.append(f"{Stage.SCRIPT.value} -> {Stage.FACTCHECK.value}")
claims_n = sum(len(line.claims) for lines in SCRIPT.values() for line in lines)
print(f"first draft: {sum(len(v) for v in SCRIPT.values())} lines, {claims_n} claims")
for position, lines in SCRIPT.items():
    display(
        Markdown(
            f"**Segment {position}**\n\n"
            + "\n".join(
                f"- **{ln.speaker}:** {ln.text}"
                + (
                    " `[cited]`"
                    if ln.claims and ln.claims[0].chunk_ids
                    else (" `[NO CITATION]`" if ln.claims else "")
                )
                for ln in lines
            )
        )
    )

## 6. Stage 5: Fact-check gate and the rewrite loop (FR9)

> *5 Fact-check: flag claims with no supporting source. Passes → voice. Unsupported claim → rewrite.*

Every claim is checked against the passages it cites. A claim fails if it **has no citation** or if
the cited passages **do not support it**. The decision of where to go next is the project's own
`route_after_factcheck`: pass → `voice`; fail with rewrites left → back to `script`; fail with none
left → held for human review (`None`), never voiced.

Offline, "supported" means at least 60% of the claim's content words appear in the cited passages. That is a
*lexical proxy*: it catches fabricated detail but not a subtle contradiction. With an LLM enabled, an
LLM judge decides instead. Neither replaces the PRD's manual claim-level audit.

In [ ]:
class Verdict(BaseModel):
    claim_number: int
    supported: bool
    reason: str


class Verdicts(BaseModel):
    verdicts: list[Verdict]


SUPPORT_THRESHOLD = 0.6


def lexical_support(claim: str, passages: list[str]) -> float:
    tokens = set(content_tokens(claim))
    if not tokens:
        return 1.0
    evidence = set(content_tokens(" ".join(passages)))
    return len(tokens & evidence) / len(tokens)


def judge_llm(items: list[tuple[Claim, list[str]]]) -> dict[int, tuple[bool, str]]:
    listing = "\n\n".join(
        f"Claim {n}: {claim.text}\nCited passages:\n" + "\n".join(f"- {p}" for p in passages)
        for n, (claim, passages) in enumerate(items, start=1)
    )
    result = LLM.parse(
        system=load_prompt(ROOT / "prompts", "factcheck", PROMPT_VERSIONS["factcheck"]),
        user=listing,
        output_format=Verdicts,
    )
    return {v.claim_number: (v.supported, v.reason) for v in result.verdicts}


def check_script(
    script: dict[int, list[ScriptLine]],
) -> tuple[FactCheckReport, dict[UUID, str], dict[UUID, int]]:
    """Judge every claim. Returns the report, a reason per failed claim and each claim's segment."""
    segment_of: dict[UUID, int] = {}
    to_judge: list[tuple[Claim, list[str]]] = []
    reasons: dict[UUID, str] = {}
    unsupported: list[Claim] = []
    total = 0
    for position, lines in script.items():
        for line in lines:
            for claim in line.claims:
                total += 1
                segment_of[claim.claim_id] = position
                if not claim.chunk_ids:
                    reasons[claim.claim_id] = "no citation"
                    unsupported.append(claim)
                else:
                    to_judge.append(
                        (
                            claim,
                            [INDEX.chunks[c].text for c in claim.chunk_ids if c in INDEX.chunks],
                        )
                    )
    verdicts = judge_llm(to_judge) if LLM_ON and to_judge else {}
    for n, (claim, passages) in enumerate(to_judge, start=1):
        if LLM_ON:
            ok, why = verdicts.get(n, (False, "no verdict returned"))
        else:
            score = lexical_support(claim.text, passages)
            ok, why = score >= SUPPORT_THRESHOLD, f"word overlap {score:.0%}"
        if not ok:
            reasons[claim.claim_id] = why
            unsupported.append(claim)
    report = FactCheckReport(
        episode_id=EPISODE_ID, unsupported=tuple(unsupported), total_claims=total
    )
    return report, reasons, segment_of


def run_gate(
    script: dict[int, list[ScriptLine]], *, verbose: bool = True
) -> tuple[FactCheckReport, list[dict]]:
    """Fact-check, then follow route_after_factcheck until the script passes (or is held)."""
    history: list[dict] = []
    rewrites_done = 0
    while True:
        with timed("5 fact-check"):
            report, reasons, segment_of = check_script(script)
        decision = route_after_factcheck(
            report, rewrites_done=rewrites_done, max_rewrites=MAX_REWRITES
        )
        label = decision.value if decision else "HELD FOR HUMAN REVIEW"
        history.append(
            {
                "pass": rewrites_done + 1,
                "claims": report.total_claims,
                "unsupported": len(report.unsupported),
                "support_rate": report.support_rate,
                "next": label,
            }
        )
        ROUTE_LOG.append(f"{Stage.FACTCHECK.value} -> {label}")
        if verbose:
            supported = report.total_claims - len(report.unsupported)
            display(
                Markdown(
                    f"**Gate pass {rewrites_done + 1}:** {supported}/{report.total_claims} claims supported ({report.support_rate:.0%}) → **{label}**"
                )
            )
            if report.unsupported:
                show_table(
                    ["Segment", "Flagged claim", "Why"],
                    [
                        [segment_of[c.claim_id], c.text, reasons[c.claim_id]]
                        for c in report.unsupported
                    ],
                )
        if decision is Stage.SCRIPT:
            redo = sorted({segment_of[c.claim_id] for c in report.unsupported})
            with timed("4 script"):
                script.update(
                    write_segments(
                        redo, attempt=rewrites_done + 1, avoid={c.text for c in report.unsupported}
                    )
                )
            ROUTE_LOG.append(f"{Stage.SCRIPT.value} -> {Stage.FACTCHECK.value}")
            rewrites_done += 1
            continue
        if decision is None:
            raise RuntimeError(
                "Rewrites exhausted: the episode is held for human review and is not voiced."
            )
        return report, history


FINAL_REPORT, FACTCHECK_HISTORY = run_gate(SCRIPT)
FIRST_DRAFT_SUPPORT = FACTCHECK_HISTORY[0]["support_rate"]

In [ ]:
# What happens when rewrites run out? Same gate, same routing function, budget of zero.
bad = FactCheckReport(
    episode_id=EPISODE_ID, unsupported=(Claim(claim_id=uuid4(), text="x"),), total_claims=4
)
routes = {
    "supported script": route_after_factcheck(
        FactCheckReport(episode_id=EPISODE_ID, unsupported=(), total_claims=4),
        rewrites_done=0,
        max_rewrites=2,
    ),
    "unsupported, rewrites left": route_after_factcheck(bad, rewrites_done=1, max_rewrites=2),
    "unsupported, rewrites spent": route_after_factcheck(bad, rewrites_done=2, max_rewrites=2),
}
show_table(
    ["Situation", "Next stage"],
    [[k, v.value if v else "None: hold for human review, never voice"] for k, v in routes.items()],
)
assert routes["unsupported, rewrites spent"] is None

## 7. Stage 6: Voice (FR6)

> *6 Voice: one TTS voice per host; reuse unchanged segments.*

Each host needs a **distinct voice**. Three backends, chosen with `TTS_BACKEND`:

| Backend | Voices | Notes |
|---|---|---|
| `say` | macOS system voices | real speech, nothing to install, Mac only |
| `chatterbox` | from `config/voices.yaml` | open model (MIT). **Can clone a voice** from a short recording |
| `placeholder` | a tone per host | rhythm and pitch only, **not speech**; for tests |

**Cloning needs consent.** The PRD lists voice cloning as a non-goal and "no cloning of real people" as a
risk mitigation, so this notebook only clones a voice whose entry in `config/voices.yaml` names the
speaker, says where their written permission is kept, and states the licence, **and** whose recording
exists. Otherwise it refuses to start and lists every problem. A host with no reference recording uses
Chatterbox's built-in default voice, which clones nobody. Only use recordings of people who agreed.

Every line is synthesised once and cached under the project's `segment_cache_key(provider, voice, text)`.
For Chatterbox the voice part of the key includes a hash of the reference recording, so swapping the
recording never serves old audio. An unchanged line is a cache hit on regeneration: that is how the PRD
keeps cost per episode low. The episode also opens with the **AI-generated disclosure, spoken** (FR13).

In [ ]:
from podcast_engine.stages.voice.registry import fingerprint, load_registry, resolve_voices


@dataclass
class VoiceStats:
    hits: int = 0
    misses: int = 0
    synth_seconds: float = 0.0


VOICE_STATS = VoiceStats()


def read_wav(path: Path) -> np.ndarray:
    with wave.open(str(path), "rb") as handle:
        if handle.getframerate() != SR or handle.getnchannels() != 1 or handle.getsampwidth() != 2:
            raise ValueError(
                f"{path.name}: need {SR} Hz mono 16-bit, got {handle.getframerate()} Hz"
            )
        frames = handle.readframes(handle.getnframes())
    return np.frombuffer(frames, dtype="<i2").astype(np.float32) / 32768.0


def write_wav(path: Path, samples: np.ndarray) -> None:
    pcm = (np.clip(samples, -1.0, 1.0) * 32767).astype("<i2")
    with wave.open(str(path), "wb") as handle:
        handle.setnchannels(1)
        handle.setsampwidth(2)
        handle.setframerate(SR)
        handle.writeframes(pcm.tobytes())


class SayBackend:
    provider = "macos-say"

    def voice_id(self, speaker: str) -> str:
        return HOSTS[speaker]

    def credit(self, speaker: str) -> str:
        return f"macOS system voice {HOSTS[speaker]}"

    def synthesise(self, speaker: str, text: str, path: Path) -> None:
        subprocess.run(
            ["say", "-v", HOSTS[speaker], "-o", str(path), f"--data-format=LEI16@{SR}", "-f", "-"],
            input=text.encode("utf-8"),
            check=True,
        )


PLACEHOLDER_PITCH_HZ = {"Alex": 200.0, "Sam": 120.0}


class PlaceholderBackend:
    """One voiced burst per word, longer for longer words. Rhythm only: this is not speech."""

    provider = "placeholder-tones"

    def voice_id(self, speaker: str) -> str:
        return f"placeholder-{speaker}"

    def credit(self, speaker: str) -> str:
        return "placeholder tone (not speech)"

    def synthesise(self, speaker: str, text: str, path: Path) -> None:
        pitch = PLACEHOLDER_PITCH_HZ.get(speaker, 150.0)
        pieces = []
        for word in text.split():
            seconds = 0.08 + 0.035 * len(word)
            t = np.arange(int(SR * seconds)) / SR
            envelope = np.sin(np.pi * t / seconds) ** 2
            wobble = 1 + 0.06 * (len(word) % 5)
            pieces.append(
                (0.3 * envelope * np.sin(2 * np.pi * pitch * wobble * t)).astype(np.float32)
            )
            pieces.append(np.zeros(int(SR * 0.04), dtype=np.float32))
        write_wav(path, np.concatenate(pieces) if pieces else np.zeros(1, dtype=np.float32))


class ChatterboxBackend:
    """Open TTS model (MIT) that can imitate a voice from a short recording.

    Voices come from config/voices.yaml. A cloned voice is refused unless its consent record is
    complete and the recording exists, so this constructor fails before any audio is made.
    Chatterbox stamps an imperceptible watermark into the audio it generates.
    """

    provider = "chatterbox"
    EXAGGERATION = 0.5
    CFG_WEIGHT = 0.5

    def __init__(self, hosts: list[str]) -> None:
        self.voices = resolve_voices(load_registry(VOICES_FILE), hosts, ROOT)
        os.environ["TQDM_DISABLE"] = "1"  # one progress bar per sentence would flood the notebook
        with warnings.catch_warnings():  # third-party deprecation noise while importing and loading
            warnings.simplefilter("ignore")
            import torch
            from chatterbox.tts import ChatterboxTTS

            device = (
                "mps"
                if torch.backends.mps.is_available()
                else "cuda"
                if torch.cuda.is_available()
                else "cpu"
            )
            print(
                f"loading Chatterbox on {device} (the first run downloads the model, a few GB)..."
            )
            self.model = ChatterboxTTS.from_pretrained(device=device)
        torch.manual_seed(SEED)  # sampling is random; the seed makes a rerun repeatable

    def voice_id(self, speaker: str) -> str:
        return f"chatterbox:{fingerprint(self.voices[speaker], ROOT)}:e{self.EXAGGERATION}:c{self.CFG_WEIGHT}"

    def credit(self, speaker: str) -> str:
        """Public wording, used in the show notes. The consent record stays in the manifest."""
        voice = self.voices[speaker]
        if not voice.is_clone:
            return "Chatterbox default voice (no real person cloned)"
        return f"voice cloned from a recording by {voice.speaker}, with their permission"

    def consent_record(self, speaker: str) -> dict[str, str]:
        voice = self.voices[speaker]
        return (
            {"speaker": voice.speaker, "consent": voice.consent, "license": voice.license}
            if voice.is_clone
            else {}
        )

    def synthesise(self, speaker: str, text: str, path: Path) -> None:
        import torchaudio

        voice = self.voices[speaker]
        reference = str(ROOT / voice.reference_audio) if voice.is_clone else None
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            wav = self.model.generate(
                text,
                audio_prompt_path=reference,
                exaggeration=self.EXAGGERATION,
                cfg_weight=self.CFG_WEIGHT,
            )
        if self.model.sr != SR:
            wav = torchaudio.functional.resample(wav, self.model.sr, SR)
        write_wav(path, wav.squeeze().cpu().numpy())


def make_backend(
    choice: str, hosts: list[str]
) -> SayBackend | PlaceholderBackend | ChatterboxBackend:
    if choice == "say":
        return SayBackend()
    if choice == "chatterbox":
        return ChatterboxBackend(hosts)
    return PlaceholderBackend()


BACKEND = make_backend(TTS_CHOICE, HOST_NAMES)


def voice_line(speaker: str, text: str) -> np.ndarray:
    key = segment_cache_key(BACKEND.provider, BACKEND.voice_id(speaker), text)  # "tts/<digest>.wav"
    path = TTS_CACHE / Path(key).name
    if path.exists():
        VOICE_STATS.hits += 1
    else:
        started = time.perf_counter()
        BACKEND.synthesise(speaker, text, path)
        VOICE_STATS.synth_seconds += time.perf_counter() - started
        VOICE_STATS.misses += 1
    return read_wav(path)

In [ ]:
check_setup_is_current()


@dataclass(frozen=True)
class Spoken:
    kind: str  # opening | disclosure | segment | closing
    position: int | None
    speaker: str
    text: str


def episode_lines(script: dict[int, list[ScriptLine]]) -> list[Spoken]:
    first, second = HOST_NAMES[0], HOST_NAMES[-1]
    opening = [
        Spoken("disclosure", None, first, AI_DISCLOSURE),
        Spoken("opening", None, first, f"Welcome to the show. Today: {SHOW.topic}."),
        Spoken("opening", None, second, "Glad to be here. Let's get into it."),
    ]
    body = [
        Spoken("segment", pos, ln.speaker, ln.text) for pos in sorted(script) for ln in script[pos]
    ]
    closing = [
        Spoken("closing", None, second, "That's all for this episode."),
        Spoken(
            "closing",
            None,
            first,
            "Thanks for listening. Every source is linked in the show notes.",
        ),
    ]
    return opening + body + closing


def stage_voice(script: dict[int, list[ScriptLine]]) -> tuple[list[Spoken], list[np.ndarray]]:
    lines = episode_lines(script)
    clips = []
    for number, line in enumerate(lines, start=1):
        misses, started = VOICE_STATS.misses, time.perf_counter()
        clips.append(voice_line(line.speaker, line.text))
        if VOICE_STATS.misses > misses:  # cached lines are instant, so only report real synthesis
            took = time.perf_counter() - started
            print(
                f"  voiced {number}/{len(lines)} {line.speaker}: {line.text[:56]!r} ({took:.0f} s)",
                flush=True,
            )
    return lines, clips


assert len({BACKEND.voice_id(h) for h in HOST_NAMES}) == len(HOST_NAMES), (
    "each host needs a distinct voice: give each host a different reference recording in config/voices.yaml"
)
with timed("6 voice"):
    LINES, CLIPS = stage_voice(SCRIPT)

ROUTE_LOG.append(f"{Stage.VOICE.value} -> {next_stage(Stage.VOICE).value}")
show_table(
    ["Host", "Voice", "Lines", "Seconds of speech"],
    [
        [
            h,
            BACKEND.voice_id(h),
            sum(1 for ln in LINES if ln.speaker == h),
            f"{sum(len(c) for ln, c in zip(LINES, CLIPS, strict=True) if ln.speaker == h) / SR:.1f}",
        ]
        for h in HOST_NAMES
    ],
)
print(
    f"voice cache ({BACKEND.provider}): {VOICE_STATS.misses} synthesised, {VOICE_STATS.hits} reused"
)

### Is this your voice?

The table above shows each host's voice id. `chatterbox:<12 characters>` is a **clone** (the characters are a
fingerprint of the recording it was cloned from) and `chatterbox:default` is Chatterbox's built-in voice. The
next cell lets you check by ear and by measurement: it plays your recording, then a line spoken by each host,
and scores how close each host's voice is to your recording (Chatterbox's own voice encoder; 1.0 = identical).

The score is the model's own idea of a voice, not an independent verdict. A clone should score clearly
higher than the other host. If it does not, the default voice may already resemble yours, or the recording
limits the clone (noise, low level, too short). Trust your ears first.

In [ ]:
def voice_check() -> None:
    if not isinstance(BACKEND, ChatterboxBackend):
        print(
            f"The voice check needs TTS_BACKEND = 'chatterbox' (this run used {BACKEND.provider})."
        )
        return
    import librosa

    def embed(samples: np.ndarray, rate: int) -> np.ndarray:
        mono = librosa.resample(samples, orig_sr=rate, target_sr=16000, res_type="soxr_hq")
        return BACKEND.model.ve.embeds_from_wavs([mono], sample_rate=16000)[0]

    clones = [h for h in HOST_NAMES if BACKEND.voices[h].is_clone]
    if not clones:
        print(
            "No host is a clone: every host uses the default voice, so there is no recording to compare with."
        )
        return
    spoken = {
        h: [c for ln, c in zip(LINES, CLIPS, strict=True) if ln.speaker == h and len(c) / SR > 3]
        for h in HOST_NAMES
    }
    voice_score = {}
    for host, clips in spoken.items():
        vectors = np.stack([embed(c, SR) for c in clips[:8]])
        mean = vectors.mean(axis=0)
        voice_score[host] = mean / np.linalg.norm(mean)

    rows, verdicts = [], []
    for owner in clones:
        reference, rate = librosa.load(
            str(ROOT / BACKEND.voices[owner].reference_audio), sr=None, mono=True
        )
        reference_vector = embed(reference, int(rate))
        half = len(reference) // 2
        natural = float(embed(reference[:half], int(rate)) @ embed(reference[half:], int(rate)))
        scores = {h: float(voice_score[h] @ reference_vector) for h in HOST_NAMES}
        for host in HOST_NAMES:
            rows.append(
                [
                    host,
                    "clone of " + BACKEND.voices[host].speaker
                    if BACKEND.voices[host].is_clone
                    else "default voice",
                    BACKEND.voices[owner].speaker,
                    f"{scores[host]:.2f}",
                ]
            )
        others = [score for host, score in scores.items() if host != owner]
        gap = scores[owner] - max(others, default=0.0)
        verdicts.append(
            f"**{owner}** scores {scores[owner]:.2f} against {BACKEND.voices[owner].speaker}'s recording"
            + (f", {gap:+.2f} versus the other host. " if others else ". ")
            + (
                "That is clearly closer: the clone is doing something."
                if gap >= 0.05
                else "That is **not clearly closer** than the other voice, so judge by ear."
            )
            + f" (The recording scores {natural:.2f} against its own other half, the ceiling for a perfect match.)"
        )
    show_table(["Host", "Voice", "Compared with the recording of", "Similarity"], rows)
    for verdict in verdicts:
        display(Markdown(verdict))

    display(Markdown("**Listen: the recording, then each host.**"))
    to_hear = [
        (f"Your recording ({BACKEND.voices[clones[0]].speaker})", reference, int(rate)),
        *[(f"{host} (first line)", spoken[host][0], SR) for host in HOST_NAMES if spoken[host]],
    ]
    for label, samples, sample_rate in to_hear:
        display(Markdown(f"*{label}*"))
        display(Audio(samples, rate=sample_rate))


voice_check()

## 8. Stage 7: Assemble (FR6, audio quality requirements)

> *7 Assemble: intro, outro, transitions; loudness normalisation.*

Lines are joined with pauses, a transition between segments, and an intro and outro. If
`assets/audio/intro.wav` or `outro.wav` exist (24 kHz mono 16-bit) they are used; otherwise a short
synthetic chime stands in. The mix is normalised to about **-16 LUFS** with `pyloudnorm`, then checked
against the PRD's audio requirements: **24 kHz or higher, no audible clipping, no long silences**.

In [ ]:
import pyloudnorm as pyln

SILENCE_LIMIT_S = 2.0
CLIP_CEILING = 0.97


def chime(freqs: tuple[float, ...], seconds: float) -> np.ndarray:
    t = np.arange(int(SR * seconds)) / SR
    mix = np.zeros_like(t)
    for n, freq in enumerate(freqs):
        onset = 0.16 * n
        envelope = np.where(t >= onset, np.exp(-(t - onset) * 2.4), 0.0)
        mix += np.sin(2 * np.pi * freq * t) * envelope
    return (0.28 * mix / len(freqs)).astype(np.float32)


def load_asset(name: str, fallback: np.ndarray) -> tuple[np.ndarray, str]:
    path = ROOT / "assets" / "audio" / f"{name}.wav"
    if path.exists():
        try:
            return read_wav(path), f"assets/audio/{name}.wav"
        except ValueError as error:
            print(f"ignoring {error}")
    return fallback, "synthetic chime"


INTRO, INTRO_SRC = load_asset("intro", chime((523.25, 659.25, 783.99, 1046.5), 2.4))
OUTRO, OUTRO_SRC = load_asset("outro", chime((1046.5, 783.99, 659.25, 523.25), 2.4))
TRANSITION = chime((880.0, 1174.66), 0.5)


def soft_limit(samples: np.ndarray, knee: float = 0.8) -> np.ndarray:
    """Smoothly squash peaks above `knee` toward CLIP_CEILING instead of hard clipping."""
    magnitude = np.abs(samples)
    room = CLIP_CEILING - knee
    squashed = knee + room * np.tanh((magnitude - knee) / room)
    return np.where(magnitude > knee, np.sign(samples) * squashed, samples).astype(np.float32)


def normalise(samples: np.ndarray) -> np.ndarray:
    """Bring loudness to the target, limiting peaks, and nudge the gain until the limiter's effect is absorbed."""
    meter = pyln.Meter(SR)
    out = samples
    gain_db = TARGET_LUFS - meter.integrated_loudness(samples)
    for _ in range(4):
        out = soft_limit(samples * 10 ** (gain_db / 20))
        error = TARGET_LUFS - meter.integrated_loudness(out)
        if abs(error) < 0.2:
            break
        gain_db += error
    return out


def silence(seconds: float) -> np.ndarray:
    return np.zeros(int(SR * seconds), dtype=np.float32)


def longest_silence(
    samples: np.ndarray, *, threshold: float = 0.003, frame_s: float = 0.02
) -> float:
    n = int(SR * frame_s)
    frames = samples[: len(samples) // n * n].reshape(-1, n)
    quiet = np.sqrt((frames**2).mean(axis=1)) < threshold
    run = best = 0
    for is_quiet in quiet:
        run = run + 1 if is_quiet else 0
        best = max(best, run)
    return best * frame_s


def audio_report(samples: np.ndarray) -> dict:
    lufs = pyln.Meter(SR).integrated_loudness(samples)
    return {
        "sample_rate_hz": SR,
        "duration_s": len(samples) / SR,
        "lufs": float(lufs),
        "peak": float(np.max(np.abs(samples))),
        "clipped_samples": int(np.sum(np.abs(samples) >= 0.999)),
        "longest_silence_s": longest_silence(samples),
    }


def stage_assemble(lines: list[Spoken], clips: list[np.ndarray]) -> tuple[np.ndarray, dict]:
    parts: list[np.ndarray] = []
    chapters: list[tuple[float, str]] = []
    cursor = 0.0
    segment_titles = {s.position: s.title for s in SEGMENTS}

    def add(signal: np.ndarray) -> None:
        nonlocal cursor
        parts.append(signal)
        cursor += len(signal) / SR

    add(INTRO)
    add(silence(0.4))
    current: int | None = None
    previous_kind = "disclosure"
    for line, clip in zip(lines, clips, strict=True):
        if line.kind == "segment" and line.position != current:
            if current is not None:
                add(silence(0.3))
                add(TRANSITION)
            add(silence(0.5))
            chapters.append((cursor, segment_titles[line.position]))
            current = line.position
        elif line.kind == "closing" and previous_kind == "segment":
            add(silence(0.3))
            add(TRANSITION)
            add(silence(0.5))
        add(clip)
        add(silence(0.7 if line.kind == "disclosure" else 0.25))
        previous_kind = line.kind
    add(silence(0.3))
    add(OUTRO)

    raw = np.concatenate(parts)
    before = audio_report(raw)
    return normalise(raw), {
        "before": before,
        "after": audio_report(normalise(raw)),
        "chapters": chapters,
    }


with timed("7 assemble"):
    EPISODE_AUDIO, ASSEMBLY = stage_assemble(LINES, CLIPS)

ROUTE_LOG.append(f"{Stage.ASSEMBLE.value} -> {next_stage(Stage.ASSEMBLE).value}")
after = ASSEMBLY["after"]
checks = [
    [
        "Sample rate",
        f"{after['sample_rate_hz']} Hz",
        f">= {MIN_SAMPLE_RATE_HZ} Hz",
        after["sample_rate_hz"] >= MIN_SAMPLE_RATE_HZ,
    ],
    [
        "Loudness",
        f"{after['lufs']:.1f} LUFS (was {ASSEMBLY['before']['lufs']:.1f})",
        f"{TARGET_LUFS:.0f} LUFS +/- 1",
        not needs_normalisation(after["lufs"]),
    ],
    [
        "Clipping",
        f"{after['clipped_samples']} samples, peak {after['peak']:.2f}",
        "none",
        after["clipped_samples"] == 0,
    ],
    [
        "Longest silence",
        f"{after['longest_silence_s']:.2f} s",
        f"< {SILENCE_LIMIT_S} s",
        after["longest_silence_s"] < SILENCE_LIMIT_S,
    ],
]
show_table(
    ["Check", "Measured", "Requirement", "Pass"],
    [[*c[:3], "PASS" if c[3] else "**FAIL**"] for c in checks],
)
AUDIO_CHECKS_PASS = all(c[3] for c in checks)
print(f"intro: {INTRO_SRC}; outro: {OUTRO_SRC}; length {after['duration_s']:.0f} s")
write_wav(RUN_DIR / "episode_r1.wav", EPISODE_AUDIO)
Audio(EPISODE_AUDIO, rate=SR)

## 9. Stage 8: Publish (FR5, FR10, FR13)

> *8 Publish: RSS feed and show notes; AI-generated label.*

Show notes list every segment's claims with **numbered source references** (FR5), chapter timestamps,
and the AI-generated label (FR13), which is also in the RSS title and description. The RSS file is
built with `feedgen` and then **parsed back with `feedparser`** to check that a podcast app would
find the episode, the enclosure and the label.

Podcast apps generally expect MP3 or M4A. This notebook publishes WAV because ffmpeg is not required to
run it; add an encode step for a real feed.

In [ ]:
import feedparser
from feedgen.feed import FeedGenerator


def source_numbers(script: dict[int, list[ScriptLine]]) -> dict[UUID, int]:
    """Number sources in order of first citation."""
    numbers: dict[UUID, int] = {}
    for position in sorted(script):
        for line in script[position]:
            for claim in line.claims:
                for chunk_id in claim.chunk_ids:
                    item_id = INDEX.chunks[chunk_id].item_id
                    numbers.setdefault(item_id, len(numbers) + 1)
    return numbers


def clock(seconds: float) -> str:
    return f"{int(seconds // 60):02d}:{int(seconds % 60):02d}"


def build_show_notes(
    script: dict[int, list[ScriptLine]], assembly: dict, revision: int, note: str = ""
) -> str:
    numbers = source_numbers(script)
    lines = [f"# {SHOW.topic}", ""]
    if note:
        lines += [f"_{note}_", ""]
    by_title = {s.title: s for s in SEGMENTS}
    for start, title in assembly["chapters"]:
        segment = by_title[title]
        lines += [f"## {clock(start)}  {title}", ""]
        for line in script[segment.position]:
            for claim in line.claims:
                refs = ", ".join(
                    f"[{numbers[INDEX.chunks[c].item_id]}]"
                    for c in claim.chunk_ids
                    if c in INDEX.chunks
                )
                lines.append(f"- {claim.text} {refs}")
        lines.append("")
    lines += ["## Sources", ""]
    lines += [
        f"{n}. [{STORE.items[i].title}]({STORE.items[i].url})"
        for i, n in sorted(numbers.items(), key=lambda kv: kv[1])
    ]
    if SOURCE_ATTRIBUTION:
        lines += ["", SOURCE_ATTRIBUTION]
    lines += [
        "",
        "## How this episode was made",
        "",
        f"- Revision {revision}, run `{RUN_ID}`, episode `{EPISODE_ID}`",
        f"- Script and fact-check: {LLM_PROVIDER + ' ' + MODEL if LLM_ON else 'offline stand-in'}; prompts {PROMPT_VERSIONS}",
        "- Voices: " + "; ".join(f"{h}: {BACKEND.credit(h)}" for h in HOST_NAMES),
        f"- Seed: {SEED}",
    ]
    return with_disclosure("\n".join(lines))


def stage_publish(
    script: dict[int, list[ScriptLine]],
    assembly: dict,
    audio_path: Path,
    revision: int,
    note: str = "",
) -> dict:
    notes = build_show_notes(script, assembly, revision, note)
    (RUN_DIR / f"show_notes_r{revision}.md").write_text(notes, encoding="utf-8")

    feed = FeedGenerator()
    feed.load_extension("podcast")
    feed.title(f"{SHOW.topic} [AI-generated]")
    feed.link(href=f"{settings.public_feed_base_url}/shows/{SHOW.show_id}", rel="alternate")
    feed.description(with_disclosure(f"An AI-generated show about {SHOW.topic}."))
    feed.language(SHOW.language)
    feed.podcast.itunes_category("Technology")
    entry = feed.add_entry()
    entry.id(str(EPISODE_ID))
    entry.guid(str(EPISODE_ID), permalink=False)
    entry.title(f"{SHOW.topic} (revision {revision}) [AI-generated]")
    entry.description(notes)
    entry.published(datetime.now(UTC))
    entry.enclosure(
        f"{settings.public_feed_base_url}/episodes/{audio_path.name}",
        str(audio_path.stat().st_size),
        "audio/wav",
    )
    feed_path = (
        RUN_DIR / "store" / feed_key(SHOW.show_id)
    )  # mirrors the object-storage key the API serves
    feed_path.parent.mkdir(parents=True, exist_ok=True)
    feed.rss_file(str(feed_path), pretty=True)

    parsed = feedparser.parse(str(feed_path))
    parsed_entry = parsed.entries[0]
    return {
        "feed_path": feed_path,
        "entries": len(parsed.entries),
        "has_enclosure": bool(parsed_entry.enclosures),
        "label_in_title": "AI-generated" in parsed_entry.title,
        "disclosure_in_description": AI_DISCLOSURE in parsed_entry.summary,
        "claims_cited": all(
            line.claims[0].chunk_ids for lines in script.values() for line in lines if line.claims
        ),
        "notes": notes,
    }


with timed("8 publish"):
    PUBLISHED = stage_publish(SCRIPT, ASSEMBLY, RUN_DIR / "episode_r1.wav", revision=1)

ROUTE_LOG.append(f"{Stage.PUBLISH.value} -> done")
show_table(
    ["RSS check", "Result"],
    [
        [k, "PASS" if PUBLISHED[k] else "**FAIL**"]
        for k in ("has_enclosure", "label_in_title", "disclosure_in_description", "claims_cited")
    ]
    + [["entries in feed", PUBLISHED["entries"]]],
)
display(Markdown(PUBLISHED["notes"]))

In [ ]:
# Optional: upload the feed where the API serves it, then read it back through the API.
# Needs the compose stack with ports published:
#   docker compose -f docker-compose.yml -f docker-compose.dev.yml up -d postgres redis s3 s3-init api
if PUSH_TO_S3:
    import httpx

    from podcast_engine.common.config import Settings
    from podcast_engine.common.storage import S3ObjectStore

    s3_settings = Settings(
        _env_file=ROOT / ".env",
        s3_endpoint_url=os.environ.get("S3_ENDPOINT_URL", "http://localhost:9000"),
    )
    store = S3ObjectStore(s3_settings)
    store.ensure_bucket()
    store.put(feed_key(SHOW.show_id), PUBLISHED["feed_path"].read_bytes(), "application/rss+xml")
    url = f"http://localhost:8000/feeds/{SHOW.show_id}/rss.xml"
    reply = httpx.get(url, timeout=5)
    print(f"GET {url} -> {reply.status_code} {reply.headers.get('content-type')}")
else:
    print("skipped (set PUSH_TO_S3 = True to upload the feed and fetch it through the API)")

## 10. Dynamic refresh: re-run only what changed (FR8)

> *Incremental regeneration re-scripts and re-voices only segments whose sources changed, and reuses
> cached audio for the rest.*

One source (*Finding the most relevant passages*) is updated upstream. The watcher notices through its content
hash, only that item is re-embedded (the other chunks come from the embedding cache), and only the
segments **tagged with that source** are re-scripted. Every other line keeps its cached audio. The
episode is then re-assembled and republished as revision 2 with the same episode id.

The outline is kept as is; a real scheduler could also re-plan when the candidate pool shifts a lot.

In [ ]:
def refresh(docs: list[dict]) -> dict:
    started = time.perf_counter()
    embed_before = (EMBEDDER.misses, EMBEDDER.hits)
    voice_before = (VOICE_STATS.misses, VOICE_STATS.hits)
    old_script = {p: [ln.text for ln in lines] for p, lines in SCRIPT.items()}

    with timed("1 ingest"):
        changed = ingest(docs, STORE, version=2)
    if not changed:  # change detection found nothing, so no stage needs to run
        return {
            "changed_items": [],
            "regenerated": [],
            "reused": list(SCRIPT),
            "reused_text_identical": True,
            "embed": (0, 0),
            "voice": (0, 0),
            "seconds": time.perf_counter() - started,
            "report": FINAL_REPORT,
            "history": FACTCHECK_HISTORY,
            "assembly": ASSEMBLY,
            "audio": EPISODE_AUDIO,
            "published": PUBLISHED,
            "skipped": True,
        }
    with timed("2 index"):
        index_items(changed)
    affected = sorted(s.position for s in SEGMENTS if set(s.source_item_ids) & set(changed))
    by_position = {s.position: s for s in SEGMENTS}
    with timed("3 retrieve+plan"):
        for position in affected:
            PASSAGES[position] = passages_for(by_position[position].source_item_ids)
    with timed("4 script"):
        SCRIPT.update(
            write_segments(affected, attempt=1, avoid=set())
        )  # attempt 1: no planted claims
    report, history = run_gate(SCRIPT, verbose=False)
    with timed("6 voice"):
        lines, clips = stage_voice(SCRIPT)
    with timed("7 assemble"):
        audio, assembly = stage_assemble(lines, clips)
    path = RUN_DIR / "episode_r2.wav"
    write_wav(path, audio)
    with timed("8 publish"):
        published = stage_publish(
            SCRIPT,
            assembly,
            path,
            revision=2,
            note=f"Updated: segment(s) {affected} refreshed from a changed source.",
        )
    new_script = {p: [ln.text for ln in lines_] for p, lines_ in SCRIPT.items()}
    return {
        "changed_items": [STORE.items[i].title for i in changed],
        "regenerated": affected,
        "reused": [p for p in SCRIPT if p not in affected],
        "reused_text_identical": all(
            new_script[p] == old_script[p] for p in SCRIPT if p not in affected
        ),
        "embed": (EMBEDDER.misses - embed_before[0], EMBEDDER.hits - embed_before[1]),
        "voice": (VOICE_STATS.misses - voice_before[0], VOICE_STATS.hits - voice_before[1]),
        "seconds": time.perf_counter() - started,
        "report": report,
        "history": history,
        "assembly": assembly,
        "audio": audio,
        "published": published,
        "skipped": False,
    }


FIRST_BUILD_SECONDS = sum(sum(v) for k, v in TIMINGS.items())
REFRESH = refresh(load_url_docs(SOURCE_URLS) if SOURCE_URLS else RAW_DOCS)

show_table(
    ["", "Result"],
    [
        [
            "Changed source",
            ", ".join(REFRESH["changed_items"]) or "none: every source is unchanged",
        ],
        ["Segments re-scripted", REFRESH["regenerated"]],
        ["Segments reused unchanged", REFRESH["reused"]],
        [
            "Items re-embedded",
            f"{len(REFRESH['changed_items'])} of {len(STORE.items)} ({REFRESH['embed'][0]} chunks)",
        ],
        ["Voice lines synthesised / from cache", f"{REFRESH['voice'][0]} / {REFRESH['voice'][1]}"],
        [
            "Fact-check",
            f"{REFRESH['history'][-1]['claims'] - REFRESH['history'][-1]['unsupported']}/{REFRESH['history'][-1]['claims']} claims supported",
        ],
        ["Time", f"{REFRESH['seconds']:.1f} s (first build: {FIRST_BUILD_SECONDS:.1f} s)"],
    ],
)
assert REFRESH["skipped"] or REFRESH["regenerated"], (
    "The updated source is not used by any segment, so nothing needed regenerating. Raise MAX_SEGMENTS."
)
assert REFRESH["reused_text_identical"], (
    "A reused segment changed: incremental regeneration is broken."
)
if REFRESH["skipped"]:
    display(Markdown("**No revision 2:** every source is unchanged, so no stage was re-run."))
else:
    display(Markdown("**Revision 2**"))
    display(Audio(REFRESH["audio"], rate=SR))

## 11. Scorecard: the PRD's success metrics and requirements

Metrics the notebook **can** measure are measured. The listener metrics (naturalness, comprehension) and
the claim-level *manual* audit need people, so they are marked as not measured rather than guessed.

In [ ]:
TOTAL_SECONDS = sum(sum(v) for v in TIMINGS.values())
episode_s = ASSEMBLY["after"]["duration_s"]


def watermark_score(samples: np.ndarray) -> float | None:
    """Chatterbox stamps an imperceptible watermark into its audio. 1.0 = detected, 0.0 = not."""
    if TTS_CHOICE != "chatterbox":
        return None
    import perth

    return float(perth.PerthImplicitWatermarker().get_watermark(samples, sample_rate=SR))


WATERMARK = watermark_score(EPISODE_AUDIO)
VOICE_SHARE = sum(TIMINGS.get("6 voice", [0.0])) / max(TOTAL_SECONDS, 1e-9)
PROJECTED_7MIN = (
    FIRST_BUILD_SECONDS * 7 / max(episode_s / 60, 0.1)
)  # the PRD's target is for 7 minutes
# USD per million tokens, (input, output), as (until 2026-12-31, from 2027-01-01). List prices read from the
# providers' pricing pages in October 2026. Verify before relying on them; thinking tokens count as output.
_GEMINI_FLASH = ((0.75, 3.75), (1.50, 7.50))
PRICES = {
    "gemini-3.8-flash": _GEMINI_FLASH,
    "gemini-3.7-flash": _GEMINI_FLASH,
    "gemini-3.5-flash": ((1.50, 9.00), (1.50, 9.00)),
    "claude-sonnet-5-5": ((2.0, 10.0), (2.0, 10.0)),
    "claude-opus-5-5": ((4.0, 20.0), (4.0, 20.0)),
    "claude-haiku-4-5": ((1.0, 5.0), (1.0, 5.0)),
}
cost = None
if LLM_ON and MODEL in PRICES:
    price_in, price_out = PRICES[MODEL][0 if NOW.year < 2027 else 1]
    cost = (LLM.usage["input_tokens"] * price_in + LLM.usage["output_tokens"] * price_out) / 1e6

stage_rows = [[k, f"{sum(v):.3f} s", len(v)] for k, v in sorted(TIMINGS.items())]
display(Markdown("**Time per stage** (whole notebook run, including the refresh)"))
show_table(["Stage", "Total time", "Runs"], stage_rows)
display(Markdown("**Route taken through the pipeline**\n\n```\n" + "\n".join(ROUTE_LOG) + "\n```"))

final = FACTCHECK_HISTORY[-1]
metrics = [
    [
        "Claims supported by a cited source",
        "90% or more",
        f"{final['support_rate']:.0%} after the gate ({FIRST_DRAFT_SUPPORT:.0%} on the first draft)",
        final["support_rate"] >= 0.9,
        "automated proxy, not the manual audit",
    ],
    [
        "Generation time",
        "under 10 min for a 7-min episode",
        f"{FIRST_BUILD_SECONDS:.0f} s for {episode_s / 60:.1f} min of audio, about {PROJECTED_7MIN:.0f} s for 7 min (refresh: {REFRESH['seconds']:.0f} s)",
        PROJECTED_7MIN < 600,
        f"projected linearly; voice synthesis is {VOICE_SHARE:.0%} of the time",
    ],
    [
        "Cost per episode",
        "under USD 1",
        f"USD {cost:.3f} ({LLM.usage['input_tokens']} in / {LLM.usage['output_tokens']} out tokens)"
        if cost is not None
        else (
            f"{LLM.usage['input_tokens']} in / {LLM.usage['output_tokens']} out tokens; no price on file for {MODEL}"
            if LLM_ON
            else "USD 0 (offline run: no paid calls)"
        ),
        None if LLM_ON and cost is None else (cost is None or cost < 1.0),
        "model calls only; hosted TTS would add cost",
    ],
    [
        "Episode length",
        "5 to 10 min",
        f"{episode_s / 60:.1f} min",
        5 <= episode_s / 60 <= 10,
        "length follows the source; raise MAX_SEGMENTS or FACTS_PER_SEGMENT for more"
        if SOURCE_URLS
        else f"sample sources hold ~{SHOW.target_minutes} min of material",
    ],
    [
        "Audio quality",
        "24 kHz+, -16 LUFS, no clipping or long silence",
        "all four checks" if AUDIO_CHECKS_PASS else "see audio table above",
        AUDIO_CHECKS_PASS,
        "",
    ],
    [
        "AI watermark in the final audio",
        "present (transparency)",
        "not applicable to this backend"
        if WATERMARK is None
        else f"detection score {WATERMARK:.1f}",
        None if WATERMARK is None else WATERMARK >= 0.5,
        "Chatterbox embeds it; checked on the mixed episode",
    ],
    [
        "Listener naturalness (MOS)",
        "3.8 or higher",
        "not measured",
        None,
        "needs a listener survey (RQ3)",
    ],
    [
        "Comprehension vs text summary",
        "equal or better",
        "not measured",
        None,
        "needs a between-group quiz (RQ2)",
    ],
]
display(Markdown("**Success metrics**"))
show_table(
    ["Metric", "Target", "Measured", "Status", "Note"],
    [
        [m[0], m[1], m[2], "n/a" if m[3] is None else ("PASS" if m[3] else "**NOT MET**"), m[4]]
        for m in metrics
    ],
)

In [ ]:
published = PUBLISHED
requirements = [
    (
        "FR1",
        "Create a show (topic, tone, length, hosts, language)",
        True,
        f"ShowConfig validated: {SHOW.host_count} hosts, {SHOW.target_minutes} min",
    ),
    (
        "FR2",
        "Ingest RSS, URLs, PDFs, search results",
        None,
        (
            f"{len(SOURCE_URLS)} web page(s) ingested as {len(RAW_DOCS)} sections; "
            if SOURCE_URLS
            else "bundled texts and optional RSS summaries; "
        )
        + "PDFs and search not covered",
    ),
    (
        "FR3",
        "Retrieve and rank relevant passages",
        True,
        f"{len(pool)} passages in pool after topic matching; 2 ranking profiles compared",
    ),
    (
        "FR4",
        "Outline, then grounded multi-speaker script",
        len(SEGMENTS) > 0,
        f"{len(SEGMENTS)} segments, {len(HOST_NAMES)} speakers",
    ),
    (
        "FR5",
        "Every claim cites a source, shown in show notes",
        published["claims_cited"],
        "citation numbers in show notes",
    ),
    (
        "FR6",
        "Distinct voice per host; intro, outro, transitions",
        len({BACKEND.voice_id(h) for h in HOST_NAMES}) == len(HOST_NAMES),
        f"voices: {[BACKEND.voice_id(h) for h in HOST_NAMES]}",
    ),
    (
        "FR7",
        "Review and edit the script before audio",
        None,
        "not covered (edit SCRIPT between the fact-check and voice cells to try it by hand)",
    ),
    (
        "FR8",
        "Regenerate only changed segments",
        None
        if REFRESH["skipped"]
        else bool(REFRESH["regenerated"]) and REFRESH["reused_text_identical"],
        "sources unchanged, so nothing was re-run (change detection only)"
        if REFRESH["skipped"]
        else f"re-scripted {REFRESH['regenerated']}, reused {REFRESH['reused']}",
    ),
    (
        "FR9",
        "Fact-check gate flags unsupported claims",
        True,
        f"first pass flagged {FACTCHECK_HISTORY[0]['unsupported']}, final pass {final['unsupported']}",
    ),
    (
        "FR10",
        "RSS feed and show notes for podcast apps",
        published["entries"] == 1 and published["has_enclosure"],
        "feed parsed back with feedparser; WAV not MP3",
    ),
    ("FR11", "Listener-level personalisation (Could)", None, "not covered"),
    ("FR12", "Interactive segments (Could)", None, "not covered"),
    (
        "FR13",
        "AI-generated disclosure",
        published["label_in_title"] and published["disclosure_in_description"],
        "spoken at the start, in show notes, RSS title and description",
    ),
]
show_table(
    ["ID", "Requirement", "Status", "Evidence"],
    [
        [r[0], r[1], "n/a" if r[2] is None else ("PASS" if r[2] else "**FAIL**"), r[3]]
        for r in requirements
    ],
)
failures = [r[0] for r in requirements if r[2] is False]
print("requirements failing:", failures or "none")

## 12. Reproducibility record

The PRD requires prompts, model versions and seeds to be stored so an experiment can be rerun. This
cell writes them, plus the results above, next to the audio.

In [ ]:
MANIFEST = {
    "run_id": RUN_ID,
    "episode_id": str(EPISODE_ID),
    "show": json.loads(SHOW.model_dump_json()),
    "script_backend": f"{LLM_PROVIDER}:{MODEL}" if LLM_ON else "offline-extractive",
    "prompt_versions": PROMPT_VERSIONS,
    "embedder": EMBEDDER.name,
    "tts_provider": BACKEND.provider,
    "voices": {
        h: {
            "voice_id": BACKEND.voice_id(h),
            "credit": BACKEND.credit(h),
            **getattr(BACKEND, "consent_record", lambda _s: {})(h),
        }
        for h in HOST_NAMES
    },
    "seed": SEED,
    "ranking_profile": RANKING_PROFILE,
    "llm_usage": dict(LLM.usage) if LLM_ON else None,
    "llm_models_used": dict(LLM.models_used) if LLM_ON else None,
}
(RUN_DIR / "run_manifest.json").write_text(json.dumps(MANIFEST, indent=2), encoding="utf-8")

RESULTS = {
    "route_log": ROUTE_LOG,
    "factcheck_history": FACTCHECK_HISTORY,
    "first_draft_support": FIRST_DRAFT_SUPPORT,
    "final_support": final["support_rate"],
    "refresh": {
        k: REFRESH[k]
        for k in (
            "changed_items",
            "regenerated",
            "reused",
            "reused_text_identical",
            "embed",
            "voice",
            "skipped",
        )
    },
    "audio": ASSEMBLY["after"],
    "audio_checks_pass": AUDIO_CHECKS_PASS,
    "watermark_score": WATERMARK,
    "voice_share_of_time": VOICE_SHARE,
    "rss": {
        k: published[k]
        for k in (
            "entries",
            "has_enclosure",
            "label_in_title",
            "disclosure_in_description",
            "claims_cited",
        )
    },
    "requirements": {r[0]: r[2] for r in requirements},
    "tts_provider": BACKEND.provider,
    "script_backend": MANIFEST["script_backend"],
    "llm_models_used": MANIFEST["llm_models_used"],
    "llm_busy_retries": getattr(LLM, "busy_retries", 0),
    "llm_fallback_model": getattr(LLM, "fallback", ""),
}
(OUTPUT / "last_results.json").write_text(
    json.dumps(RESULTS, indent=2, default=str), encoding="utf-8"
)
print("wrote", shown(RUN_DIR / "run_manifest.json"), "and", shown(OUTPUT / "last_results.json"))